## NB03: Data Analysis

In [8]:
import kaleido
import pandas as pd
import plotly.express as px

In [9]:
movies_df = pd.read_csv("../data/processed/movies.csv")
genres_df = pd.read_csv("../data/processed/genres.csv")
movies_with_genres_df = pd.read_csv("../data/processed/movies_with_genres.csv")

### Chart 1 — Bar chart, average popularity by genre

In [10]:
genre_stats = (movies_with_genres_df.groupby("genre").agg(n_movies=("id","nunique"), avg_popularity=("popularity","mean")) ).reset_index().sort_values("avg_popularity", ascending=False)

fig1 = px.bar(genre_stats, x="genre", y="avg_popularity",
             hover_data=["n_movies"], color="avg_popularity",color_continuous_scale="Tealgrn")
fig1.update_layout(title="Sci-Fi and Adventure films draw far more attention than Drama, despite being rarer")
fig1.write_html("../docs/charts/finding1_average_popularity_by_genre.html", include_plotlyjs="cdn", full_html=True)
fig1.show()

Results: Science Fiction has the highest average popularity (14.13), followed by Adventure (13.24) and Fantasy (12.32). Action (11.50) and Animation (10.09) also rank above the overall midpoint. Drama, despite being the most frequent genre in the dataset (3041 movies), has one of the lowest average popularity scores (7.51), tied with Romance. Documentary sits at the bottom (5.42). This suggests that how often a genre appears in the dataset is not related to how much attention it draws.

### Chart 2 — Scatter plot, popularity vs. vote average by genre

In [11]:
fig2 = px.scatter(genre_stats.merge(
                    movies_with_genres_df.groupby("genre")["vote_average"].mean().reset_index(),
                    on="genre"),
                  x="avg_popularity", y="vote_average", size="n_movies",
                  text="genre", hover_name="genre",color="genre")
fig2.update_layout(title="Popularity and critical rating don't move together across genres")
fig2.write_html("../docs/charts/finding2_popularityvsvote_by_decade.html", include_plotlyjs="cdn", full_html=True)
fig2.show()

Results: Popularity and rating do not move together. Animation has the highest average rating (7.14) despite only moderate popularity, while Science Fiction and Adventure — the two most popular genres — sit in the middle of the rating range (6.63 and 6.71). Horror is the clearest outlier: it is fairly popular (9.78) but has the lowest average rating among major genres (6.24). Drama, the largest genre by count (3041 movies), has an average, unremarkable rating (6.71).

### Chart 3 — Line chart, genre share of top films by decade

In [12]:
top_genres = movies_with_genres_df["genre"].value_counts().head(6).index.tolist()  # .index extracts just the genre names from value_counts(), not the counts
sub = movies_with_genres_df[
    movies_with_genres_df["genre"].isin(top_genres)  # .isin() returns True/False for each row, checking if its genre is in top_genres
    & (movies_with_genres_df["decade"] >= 1970)
]
counts = (
    sub.groupby(["decade", "genre"])
       .size()
       .rename("n")  # names the unnamed count column from .size() so it can be referenced as "n"
       .reset_index()  # turns the multi-level (decade, genre) index back into regular columns
)
counts["share_pct"] = (
    100 * counts["n"]
    / counts.groupby("decade")["n"].transform("sum")  # .transform("sum") sums "n" per decade but keeps one row per original row, so it can be divided row-by-row
)
fig3 = px.line(counts, x="decade", y="share_pct", color="genre", markers=True)
fig3.update_layout(
    title="Action-Adventure-Animation-Fantasy is the best-rated genre combination<br>in the sample, beating any single genre alone"
)
fig3.write_html("../docs/charts/finding3_genre_share_by_decade.html", include_plotlyjs="cdn", full_html=True)
fig3.show()

Results: Drama's share has fallen steadily, from 37.2% of top films in the 1970s to 24.9% in the 2020s. Action has more than doubled its share over the same period, from 9.7% to 18.5%, and Thriller has followed a similar pattern (12.5% to 18.6%). Comedy has also declined somewhat (23.6% to 18.1%), while Adventure and Romance have stayed comparatively flat across the decades. The overall pattern points to action-oriented genres steadily displacing Drama and Comedy in the composition of popular films.